# Pipeline

In [1]:
from pathlib import Path
import pandas as pd

from lana_nlp.pipeline.data_loader import LyricsDataLoader
from lana_nlp.pipeline.text_cleaner import TextCleaner
from lana_nlp.pipeline.create_features import LyricsAnalyzer

## Load the lyrics

In [2]:
loader = LyricsDataLoader(
    Path("../data/raw/lyrics.csv")
)

df = loader.load()

print(df.shape)
df.head()

(154, 5)


,artist,album,song,year,lyrics
0,Lana Del Rey,Sirens,Drive By,2006,There was a drive-by Sunday night\nMost of us ...
1,Lana Del Rey,Sirens,River Road,2006,"Another day is over, another day is done\nAnd ..."
2,Lana Del Rey,Sirens,A Star for Nick,2006,"Well, you know it and I know it, I'm gonna be ..."
3,Lana Del Rey,Sirens,My Momma,2006,My momma wouldn't say you were a nice guy\nBut...
4,Lana Del Rey,Sirens,Bad Disease,2006,"Well, there's somethin' about watchin' a crime..."


## Clean the lyrics (basic)

In [3]:
cleaner = TextCleaner()

df["basic_cleaned_lyrics"] = df["lyrics"].apply(cleaner.basic_clean)
df.head()

,artist,album,song,year,lyrics,basic_cleaned_lyrics
0,Lana Del Rey,Sirens,Drive By,2006,There was a drive-by Sunday night\nMost of us ...,there was a driveby sunday night\nmost of us w...
1,Lana Del Rey,Sirens,River Road,2006,"Another day is over, another day is done\nAnd ...",another day is over another day is done\nand n...
2,Lana Del Rey,Sirens,A Star for Nick,2006,"Well, you know it and I know it, I'm gonna be ...",well you know it and i know it I am gonna be a...
3,Lana Del Rey,Sirens,My Momma,2006,My momma wouldn't say you were a nice guy\nBut...,my momma would not say you were a nice guy\nbu...
4,Lana Del Rey,Sirens,Bad Disease,2006,"Well, there's somethin' about watchin' a crime...",well theres somethin about watchin a crime\nth...


## NLP cleaned lyrics

In [4]:
df["nlp_cleaned_lyrics"] = df["basic_cleaned_lyrics"].apply(cleaner.nlp_clean)
df.head()

,artist,album,song,year,lyrics,basic_cleaned_lyrics,nlp_cleaned_lyrics
0,Lana Del Rey,Sirens,Drive By,2006,There was a drive-by Sunday night\nMost of us ...,there was a driveby sunday night\nmost of us w...,"[driveby, sunday, night, u, bed, right, turned..."
1,Lana Del Rey,Sirens,River Road,2006,"Another day is over, another day is done\nAnd ...",another day is over another day is done\nand n...,"[another, day, another, day, done, gettin, clo..."
2,Lana Del Rey,Sirens,A Star for Nick,2006,"Well, you know it and I know it, I'm gonna be ...",well you know it and i know it I am gonna be a...,"[well, know, know, gon, star, far, say, hello,..."
3,Lana Del Rey,Sirens,My Momma,2006,My momma wouldn't say you were a nice guy\nBut...,my momma would not say you were a nice guy\nbu...,"[momma, would, say, nice, guy, 40, job, momma,..."
4,Lana Del Rey,Sirens,Bad Disease,2006,"Well, there's somethin' about watchin' a crime...",well theres somethin about watchin a crime\nth...,"[well, there, somethin, watchin, crime, make, ..."


In [5]:
df.columns

Index(['artist', 'album', 'song', 'year', 'lyrics', 'basic_cleaned_lyrics',
       'nlp_cleaned_lyrics'],
      dtype='str')

## Create Analyzers

In [6]:
analyzer = LyricsAnalyzer(
    df,
    basic_text_column="basic_cleaned_lyrics",
    nlp_text_column="nlp_cleaned_lyrics",
)

df = analyzer.analyze()

In [7]:
df.shape

(154, 12)

In [8]:
df.columns.tolist()

['artist',
 'album',
 'song',
 'year',
 'lyrics',
 'basic_cleaned_lyrics',
 'nlp_cleaned_lyrics',
 'word_count',
 'unique_words',
 'syllable_count',
 'line_count',
 'reading_minutes']

In [9]:
df.head()

,artist,album,song,year,lyrics,basic_cleaned_lyrics,nlp_cleaned_lyrics,word_count,unique_words,syllable_count,line_count,reading_minutes
0,Lana Del Rey,Sirens,Drive By,2006,There was a drive-by Sunday night\nMost of us ...,there was a driveby sunday night\nmost of us w...,"[driveby, sunday, night, u, bed, right, turned...",251,95,298,37,1.255
1,Lana Del Rey,Sirens,River Road,2006,"Another day is over, another day is done\nAnd ...",another day is over another day is done\nand n...,"[another, day, another, day, done, gettin, clo...",172,65,217,30,0.860
2,Lana Del Rey,Sirens,A Star for Nick,2006,"Well, you know it and I know it, I'm gonna be ...",well you know it and i know it I am gonna be a...,"[well, know, know, gon, star, far, say, hello,...",92,45,106,15,0.460
3,Lana Del Rey,Sirens,My Momma,2006,My momma wouldn't say you were a nice guy\nBut...,my momma would not say you were a nice guy\nbu...,"[momma, would, say, nice, guy, 40, job, momma,...",303,112,346,37,1.515
4,Lana Del Rey,Sirens,Bad Disease,2006,"Well, there's somethin' about watchin' a crime...",well theres somethin about watchin a crime\nth...,"[well, there, somethin, watchin, crime, make, ...",238,107,280,40,1.190


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 154 entries, 0 to 153
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   artist                154 non-null    str    
 1   album                 154 non-null    str    
 2   song                  154 non-null    str    
 3   year                  154 non-null    int64  
 4   lyrics                153 non-null    str    
 5   basic_cleaned_lyrics  154 non-null    str    
 6   nlp_cleaned_lyrics    154 non-null    object 
 7   word_count            154 non-null    int64  
 8   unique_words          154 non-null    int64  
 9   syllable_count        154 non-null    int64  
 10  line_count            154 non-null    int64  
 11  reading_minutes       154 non-null    float64
dtypes: float64(1), int64(5), object(1), str(5)
memory usage: 14.6+ KB


In [11]:
df.to_csv("../data/processed/lyrics.csv")

In [12]:
album_summary = analyzer.statistics.summary_by_album()
album_summary.head(11)


,songs,avg_words,median_words,min_words,max_words,total_words,avg_reading_minutes
album,,,,,,,
Ultraviolence,16,289.000000,276.5,128,428,4624,1.445000
Lust for Life,16,372.437500,362.0,219,630,5959,1.862188
Did You Know That There's a Tunnel Under Ocean Blvd,16,397.187500,355.5,199,784,6355,1.985937
Blue Banisters,15,290.333333,314.0,0,452,4355,1.451667
Born To Die,15,421.066667,412.0,277,628,6316,2.105333
Sirens,15,220.400000,238.0,92,303,3306,1.102000
Norman Fucking Rockwell!,14,346.714286,326.5,202,507,4854,1.733571
Honeymoon,14,280.214286,279.5,84,444,3923,1.401071
"Lana Del Ray, A.K.A. Lizzy Grant",13,245.615385,213.0,84,427,3193,1.228077


### Do Ultraviolence and Honeymoon have more instrumental sections?

In [13]:
# Calculate words per line column
df["words_per_line"] = (
    df["word_count"] / df["line_count"].replace(0, pd.NA)
)

df.groupby("album")["words_per_line"].mean().sort_values()

album
Paradise                                               5.808429
Lana Del Ray, A.K.A. Lizzy Grant                       5.900705
Lust for Life                                          6.323069
Blue Banisters                                         6.340419
Honeymoon                                              6.346165
Chemtrails Over the Country Club                       6.423123
Ultraviolence                                          6.450731
Did You Know That There's a Tunnel Under Ocean Blvd    6.455239
Norman Fucking Rockwell!                               6.774465
Sirens                                                 7.122797
Born To Die                                            7.464082
Name: words_per_line, dtype: object

In [14]:
df.columns.tolist()

['artist',
 'album',
 'song',
 'year',
 'lyrics',
 'basic_cleaned_lyrics',
 'nlp_cleaned_lyrics',
 'word_count',
 'unique_words',
 'syllable_count',
 'line_count',
 'reading_minutes',
 'words_per_line']